# BEACON — pan-cancer compact-BDP co-driver Navigator
## Figure reproduction notebook

**BEACON** = *Bidirectional-promoter Expression And Copy-number Output Navigator*

Reproduces the four main BEACON figures. The narrative follows three movements:

1. **Validate in Caco-2 / CRC** — compact bidirectional promoters (BDPs) are broadly coordinated in colorectal tumors, and the PLAGL2–POFUT1 locus (the subject of the companion single-locus study) ranks first. This sets the tone that compact BDPs matter.
2. **Expand to more BDPs** — move from the 505 BDPs testable in Caco-2 to the full **792 architecture-defined** universe (adding a 287-BDP *expansion pack* silent in Caco-2), removing cell-line ascertainment bias.
3. **Expand across cancers** — deploy the measures across 14 TCGA cohorts.

### Data model — per (BDP, cancer)
| column | meaning |
|---|---|
| `coord_rho` | coordinated output: Spearman ρ between the two arms' expression across tumors |
| `dosage_rho` | dosage coupling: Spearman ρ between locus copy number (GISTIC) and expression (mean of arms) |
| `amp_frac` | amplification prevalence: fraction of tumors with GISTIC ≥ +1 (mean of arms) |
| `in_caco2_505` | True = testable in Caco-2 (505); False = expansion pack (287) |
| `caco2_coupling` | Caco-2 nascent-RNA partial coupling (from the atlas; NaN for expansion pack) |
| **co-driver call** | `dosage_rho ≥ 0.3 AND amp_frac ≥ 0.15` |

> Predictions are **measured** associations, not a fitted model — hypotheses to test, not a classifier score.


## 0 · Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib as mpl
from scipy.stats import spearmanr
from matplotlib.lines import Line2D

mpl.rcParams.update({
    "figure.dpi":110,"savefig.dpi":300,"font.size":8,"axes.titlesize":8.5,
    "axes.labelsize":7,"xtick.labelsize":6,"ytick.labelsize":6,"legend.fontsize":6.5,
    "axes.spines.top":False,"axes.spines.right":False,"axes.linewidth":0.8,
    "font.family":"sans-serif","savefig.bbox":"tight"})
def panel_letter(ax,l): ax.text(-0.14,1.06,l,transform=ax.transAxes,fontsize=11,fontweight="bold",va="top")

TYPES=["coadread","stad","gbm","ucec","blca","lgg","luad","paad","brca","laml","hnsc","lihc","kirc","thca"]
CT=[t.upper() for t in TYPES]
LABELS={"COADREAD":"CRC","STAD":"Stomach","GBM":"GBM","UCEC":"Endom.","BLCA":"Bladder","LGG":"LGG",
        "LUAD":"Lung","PAAD":"Panc.","BRCA":"Breast","LAML":"AML","HNSC":"H&N","LIHC":"Liver",
        "KIRC":"Kidney","THCA":"Thyroid"}

## 1 · Load the precompute

**Option A (default):** load the shipped table `data/beacon_pancancer_precompute.csv`.
**Option B (§1b):** re-fetch from cBioPortal to regenerate it (~10 min).

In [ ]:
pan = pd.read_csv("data/beacon_pancancer_precompute.csv")
pan["codriver"] = (pan.dosage_rho>=0.3) & (pan.amp_frac>=0.15) & pan.coord_rho.notna()
print(pan.shape, "|", pan.cancer.nunique(), "cancers |", pan.bdp_id.nunique(), "BDPs")
pan.head(3)

### 1b · (optional) Rebuild the precompute from cBioPortal

The universe is the 792 compact protein-coding bidirectional promoters (≤1 kb TSS–TSS spacing)
from the census (`data/universe792.csv`). For each of 14 `*_tcga_pan_can_atlas_2018` studies we fetch
RSEM expression and GISTIC copy number for both arms, then compute the three measures.
Large responses can truncate — genes are fetched in 500-id chunks.

In [ ]:
import urllib.request, json, gc, time
API="https://www.cbioportal.org/api"
def post(path,payload):
    req=urllib.request.Request(f"{API}{path}",data=json.dumps(payload).encode(),
        headers={"Content-Type":"application/json","Accept":"application/json"},method="POST")
    with urllib.request.urlopen(req,timeout=300) as r: return json.loads(r.read().decode())

RUN_FETCH=False   # set True to actually re-fetch (~10 min, needs network)
if RUN_FETCH:
    u792=pd.read_csv("data/universe792.csv")
    genes=sorted(set(u792.geneA)|set(u792.geneB))
    g=post("/genes/fetch",genes,)  # HUGO->entrez; see repo methods for geneIdType
    sym2ent={x["hugoGeneSymbol"]:x["entrezGeneId"] for x in g}
    ent2sym={v:k for k,v in sym2ent.items()}; entrez=sorted(ent2sym)
    def fetch_matrix(prof):
        slid=prof.rsplit("_rna_seq",1)[0].rsplit("_gistic",1)[0]+"_all"; parts=[]
        for i in range(0,len(entrez),500):
            recs=post(f"/molecular-profiles/{prof}/molecular-data/fetch",
                      {"entrezGeneIds":entrez[i:i+500],"sampleListId":slid})
            parts.append(pd.DataFrame([(r["entrezGeneId"],r["sampleId"],r["value"]) for r in recs],columns=["ent","sid","val"]))
        df=pd.concat(parts,ignore_index=True); df["gene"]=df.ent.map(ent2sym); df["sid"]=df.sid.str[:15]
        return df.pivot_table(index="gene",columns="sid",values="val",aggfunc="first")
    # ... per-study loop computes coord/dosage/amp exactly as in ANALYSIS/ (see repo methods)
    print("see repo ANALYSIS scripts for the full loop")
else:
    print("Using shipped precompute (Option A).")

## 2 · Movement 1 — Validate in Caco-2 / CRC  → **Figure 1**

Compact BDPs are broadly coordinated in CRC, and PLAGL2–POFUT1 ranks first (reproducing the
single-locus study's rank-1/63 finding). Panel b shows Caco-2 nascent coupling vs CRC tumor
coordination — weak at genome scale (the honest caveat); the locus-level reproduction is the strong validation.

In [ ]:
crc505 = (pan[(pan.cancer=="COADREAD") & pan.coord_rho.notna() & pan.in_caco2_505]
          .sort_values("coord_rho",ascending=False).reset_index(drop=True))
pp_rank = int(crc505.index[crc505.geneA=="PLAGL2"][0])+1
pp = crc505[crc505.geneA=="PLAGL2"].iloc[0]
val = crc505.dropna(subset=["caco2_coupling","coord_rho"])
rr,pv = spearmanr(val.caco2_coupling, val.coord_rho)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.4))
ax1.hist(crc505.coord_rho,bins=40,color="#c6dbef",edgecolor="#6baed6",lw=0.4)
ax1.axvline(0.3,ls="--",lw=0.8,color="#999"); ax1.axvline(pp.coord_rho,color="#b2182b",lw=1.6)
ax1.annotate(f"PLAGL2–POFUT1\nrank 1/{len(crc505)}\nρ={pp.coord_rho:.2f}",
             xy=(pp.coord_rho,ax1.get_ylim()[1]*0.55),xytext=(0.35,ax1.get_ylim()[1]*0.72),
             fontsize=7,color="#b2182b",arrowprops=dict(arrowstyle="->",lw=0.8,color="#b2182b"))
ax1.text(0.31,ax1.get_ylim()[1]*0.94,"coordinated\n(ρ>0.3)",fontsize=6,color="#666")
ax1.set_xlabel("coordinated output in CRC (arm–arm ρ)"); ax1.set_ylabel("compact BDPs")
ax1.set_title(f"Compact BDPs are broadly coordinated in CRC\n{100*(crc505.coord_rho>0.3).mean():.0f}% with ρ>0.3; PLAGL2–POFUT1 ranks first",loc="left",fontsize=8.5)
panel_letter(ax1,"a")
ax2.scatter(val.caco2_coupling,val.coord_rho,s=12,c="#6baed6",alpha=0.55,edgecolors="none")
ax2.scatter(pp.caco2_coupling,pp.coord_rho,s=55,facecolors="none",edgecolors="#b2182b",lw=1.6,zorder=5)
ax2.annotate("PLAGL2–POFUT1",xy=(pp.caco2_coupling,pp.coord_rho),xytext=(pp.caco2_coupling-0.15,pp.coord_rho+0.02),fontsize=6.5,color="#b2182b",ha="right")
z=np.polyfit(val.caco2_coupling,val.coord_rho,1); xs=np.array([val.caco2_coupling.min(),val.caco2_coupling.max()])
ax2.plot(xs,z[0]*xs+z[1],color="#08519c",lw=1.3)
ax2.set_xlabel("Caco-2 nascent-RNA coupling (partial ρ)"); ax2.set_ylabel("CRC tumor coordination (ρ)")
ax2.set_title(f"Caco-2 coupling weakly concords with CRC coordination\nρ={rr:.2f}, p={pv:.1e}, n={len(val)} — locus-level reproduction is the strong validation",loc="left",fontsize=8.0)
panel_letter(ax2,"b")
fig.tight_layout(); fig.savefig("figures/BEACON_F1_validation.png",dpi=300); plt.show()

## 3 · Movement 2 — Expand to more BDPs  → **Figure 2**

The 792-BDP universe is defined by architecture alone. The 287-BDP expansion pack (silent in Caco-2)
recovers co-drivers a Caco-2-only list would miss — panel b names the top ones (note PLAG1–CHCHD7; PLAG1 is a PLAGL2 paralog).

In [ ]:
cd_any = pan[pan.codriver].groupby(["bdp_id","in_caco2_505"]).size().reset_index(name="n")
u792 = pd.read_csv("data/universe792.csv")
n_ep=int((~u792.in_caco2_505).sum()); n_c5=int(u792.in_caco2_505.sum())
hit=[cd_any[cd_any.in_caco2_505].bdp_id.nunique(), cd_any[~cd_any.in_caco2_505].bdp_id.nunique()]
tot=[n_c5,n_ep]; x=np.arange(2)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.6))
ax1.bar(x,tot,color="#e5e5e5",width=0.6,label="in 792 universe")
ax1.bar(x,hit,color=["#08519c","#a8600a"],width=0.6,label="co-driver in ≥1 cancer")
for i,(h,tt) in enumerate(zip(hit,tot)): ax1.text(i,h+6,f"{h}\n({100*h/tt:.0f}%)",ha="center",fontsize=7,fontweight="bold")
ax1.set_xticks(x); ax1.set_xticklabels(["Caco-2 testable\n(505)","expansion pack\n(287)"]); ax1.set_ylabel("compact pc–pc BDPs")
ax1.set_title("Architecture-defined universe removes Caco-2 bias\nexpansion pack recovers co-drivers a Caco-2 list misses",loc="left",fontsize=8.5)
ax1.legend(frameon=False,fontsize=6.5,loc="upper right"); panel_letter(ax1,"a")
ep=pan[(~pan.in_caco2_505)&pan.codriver]
top=ep.sort_values("coord_rho",ascending=False).drop_duplicates("bdp_id").head(12)[["bdp_id","coord_rho","dosage_rho","amp_frac"]].sort_values("coord_rho")
yp=np.arange(len(top))
ax2.hlines(yp,0,top.coord_rho,color="#e0cba8",lw=1)
sc=ax2.scatter(top.coord_rho,yp,s=np.clip(top.amp_frac*400,25,220),c=top.dosage_rho,cmap="viridis",vmin=0.3,vmax=0.7,zorder=3,edgecolors="white",lw=0.8)
ax2.set_yticks(yp); ax2.set_yticklabels(top.bdp_id,fontsize=6.5)
ax2.set_xlabel("coordinated output (arm–arm ρ)"); ax2.set_xlim(0,0.85)
cb=fig.colorbar(sc,ax=ax2,fraction=0.046,pad=0.02); cb.set_label("dosage ρ",fontsize=7)
for i,b in enumerate(top.bdp_id):
    if b=="PLAG1-CHCHD7": ax2.annotate("PLAG1 = PLAGL2 paralog",xy=(top.coord_rho.iloc[i],i),xytext=(0.08,i+0.6),fontsize=6,arrowprops=dict(arrowstyle="->",lw=0.6))
ax2.set_title("Top expansion-pack co-drivers\n(silent in Caco-2; co-driver in cancer)",loc="left",fontsize=8.5); panel_letter(ax2,"b")
fig.tight_layout(); fig.savefig("figures/BEACON_F2_expansion.png",dpi=300); plt.show()

## 4 · Movement 3a — Expand across cancers  → **Figure 3**

The co-driver landscape across 14 TCGA cohorts. The co-driver cloud (blue, shaded zone) shrinks in
copy-number-quiet cancers; PLAGL2–POFUT1 (red ring) sits high in CRC and drifts down-left elsewhere.

In [ ]:
fig,axes=plt.subplots(2,7,figsize=(14,4.6),sharex=True,sharey=True)
order=["blca","luad","brca","stad","hnsc","coadread","lihc","ucec","gbm","paad","kirc","lgg","laml","thca"]
for ax,t in zip(axes.flat,order):
    C=t.upper(); sub=pan[(pan.cancer==C)&pan.coord_rho.notna()]
    cd=sub[sub.codriver]; nc=sub[~sub.codriver]
    ax.axvspan(0.3,0.85,color="#08519c",alpha=0.05)
    ax.scatter(nc.dosage_rho,nc.coord_rho,s=5,c="#ccc",alpha=0.5,edgecolors="none")
    ax.scatter(cd.dosage_rho,cd.coord_rho,s=6,c="#08519c",alpha=0.65,edgecolors="none")
    p=sub[sub.geneA=="PLAGL2"]
    if len(p): ax.scatter(p.dosage_rho,p.coord_rho,s=42,facecolors="none",edgecolors="#b2182b",lw=1.4,zorder=5)
    ax.set_title(f"{LABELS[C]}\n{len(cd)}/{len(sub)} co-driver",fontsize=7,loc="left")
    ax.set_xlim(-0.5,0.85); ax.set_ylim(-0.5,1.0); ax.axvline(0.3,ls="--",lw=0.6,color="#999")
for ax in axes[:,0]: ax.set_ylabel("coord ρ",fontsize=7)
for ax in axes[1,:]: ax.set_xlabel("dosage ρ",fontsize=7)
fig.suptitle("Compact-BDP co-driver landscape across 14 cancers — co-driver zone shaded, PLAGL2–POFUT1 ringed (red)",fontsize=9,x=0.01,ha="left",y=1.02)
fig.tight_layout(); fig.savefig("figures/BEACON_F3_landscape.png",dpi=300); plt.show()

## 5 · Movement 3b — Scaling & burden  → **Figure 4**

Panel a: PLAGL2–POFUT1 coordinated output scales with 20q dosage across cancers (highest in CRC, lost where 20q is not amplified).
Panel b: each cancer's co-driver burden tracks its overall amplification landscape (ρ≈0.99) — copy-number-quiet cancers (AML, thyroid) yield no co-drivers.

In [ ]:
pp_all=pan[pan.geneA=="PLAGL2"].set_index("cancer").reindex(CT)
burden=(pan[pan.coord_rho.notna()].groupby("cancer")
        .agg(n=("bdp_id","nunique"),ncd=("codriver","sum"),mean_amp=("amp_frac","mean")).reindex(CT))
burden["frac_cd"]=burden.ncd/burden.n

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.4))
d=pp_all.dropna(subset=["coord_rho"]).sort_values("coord_rho"); yp=np.arange(len(d))
cols=["#08519c" if v else "#bbb" for v in ((d.dosage_rho>=0.3)&(d.amp_frac>=0.15))]
ax1.hlines(yp,0,d.coord_rho,color="#ccc",lw=1)
ax1.scatter(d.coord_rho,yp,s=np.clip(d.amp_frac*400,20,220),c=cols,zorder=3,edgecolors="white",lw=0.8)
ax1.set_yticks(yp); ax1.set_yticklabels([LABELS[c] for c in d.index]); ax1.set_xlim(0,1.0)
ax1.set_xlabel("PLAGL2–POFUT1 coordinated output (ρ)")
ax1.set_title("PLAGL2–POFUT1 scales with 20q dosage across cancers\nCRC highest; lost where 20q is not amplified",loc="left",fontsize=8.0)
ax1.legend(handles=[Line2D([0],[0],marker="o",color="w",markerfacecolor="#08519c",markersize=8,label="co-driver"),
                    Line2D([0],[0],marker="o",color="w",markerfacecolor="#bbb",markersize=8,label="no clear co-driver")],
           frameon=False,fontsize=6.5,loc="lower right"); panel_letter(ax1,"a")
bd=burden.dropna(); rr2,pv2=spearmanr(bd.mean_amp,bd.frac_cd)
ax2.scatter(bd.mean_amp,bd.frac_cd*100,s=45,c="#08519c",zorder=3,edgecolors="white",lw=0.8)
off={"THCA":None,"LAML":None,"STAD":(4,-9,"left"),"HNSC":(4,4,"left"),"BLCA":(-4,5,"right"),
     "LUAD":(4,-8,"left"),"LGG":(6,-2,"left"),"KIRC":(4,2,"left"),"GBM":(6,-2,"left"),"PAAD":(-4,-9,"right")}
for c in bd.index:
    o=off.get(c,(4,2,"left"))
    if o is None: continue
    dx,dy,ha=o; ax2.annotate(LABELS[c],xy=(bd.loc[c,"mean_amp"],bd.loc[c,"frac_cd"]*100),xytext=(dx,dy),textcoords="offset points",fontsize=6,ha=ha)
ax2.annotate("AML, Thyroid\n(0 co-drivers)",xy=(bd.loc["LAML","mean_amp"],0),xytext=(bd.loc["LAML","mean_amp"]+0.02,10),fontsize=6,ha="left",arrowprops=dict(arrowstyle="->",lw=0.6))
z=np.polyfit(bd.mean_amp,bd.frac_cd*100,1); xs=np.array([bd.mean_amp.min(),bd.mean_amp.max()])
ax2.plot(xs,z[0]*xs+z[1],color="#b2182b",lw=1.4)
ax2.set_xlabel("mean amplification prevalence (per cancer)"); ax2.set_ylabel("BDPs called co-driver (%)")
ax2.set_title(f"Co-driver burden tracks each cancer's amplification landscape\nρ={rr2:.2f}, p={pv2:.1e}, n={len(bd)} cancers",loc="left",fontsize=8.0)
ax2.margins(0.14); panel_letter(ax2,"b")
fig.tight_layout(); fig.savefig("figures/BEACON_F4_scaling_burden.png",dpi=300); plt.show()

## 6 · Summary table — per-cancer co-driver burden

In [ ]:
summary = (pan[pan.coord_rho.notna()].groupby("cancer")
           .agg(n_BDP=("bdp_id","nunique"), n_codriver=("codriver","sum"),
                mean_amp=("amp_frac","mean")).reindex(CT))
summary["pct_codriver"]=(100*summary.n_codriver/summary.n_BDP).round(1)
summary["cancer"]=[LABELS[c] for c in summary.index]
summary[["cancer","n_BDP","n_codriver","pct_codriver","mean_amp"]].round(3)